In [231]:
import math
import random

In [232]:
def parse_fleet(file):
    with open(file,"r") as f:
        lines = [line.strip() for line in f if line.strip()]
    i=0
    while i<len(lines):
        parts=lines[i].split()
        if "cvrp" in file:
            if len(parts)==1:
                vehcile_capacity=int(parts[0])
                coordinates=lines[i+1].split()
                x_coordinate=float(coordinates[0])
                y_coordinate=float(coordinates[1])
                customers=[]
                for j in range(i+2,len(lines)):
                    each_line=lines[j].split()
                    customers.append({
                        "id": j-2,
                        "x_coordinate":float(each_line[0]),
                        "y_coordinate":float(each_line[1]),
                        "demand":int(each_line[2])
                    })
                return {
                    "vehicle_capacity":vehcile_capacity,
                    "depot_coordinates":(x_coordinate,y_coordinate),
                    "customers":customers
                }
        if "vrptw" in file:
          if len(parts)==2:
                number_of_vehicles=int(parts[0])
                vehcile_capacity=int(parts[1])
                coordinates=lines[i+1].split()
                x_coordinate=float(coordinates[0])
                y_coordinate=float(coordinates[1])
                customers=[]
                for j in range(i+2,len(lines)):
                    each_line=lines[j].split()
                    customers.append({
                        "id": j-2,
                        "x_coordinate":float(each_line[0]),
                        "y_coordinate":float(each_line[1]),
                        "demand":int(each_line[2]),
                        "time_window_opening":float(each_line[3]),
                        "time_window_closing":float(each_line[4]),
                        "service_time": float(each_line[5])
                    })
                return {
                    "number_of_vehicles": number_of_vehicles,
                    "vehicle_capacity":vehcile_capacity,
                    "depot_coordinates":(x_coordinate,y_coordinate),
                    "customers":customers
                }
                
        i+=1
    return None
print(parse_fleet("vrptw.txt"))
        

{'number_of_vehicles': 25, 'vehicle_capacity': 200, 'depot_coordinates': (35.0, 35.0), 'customers': [{'id': 0, 'x_coordinate': 41.0, 'y_coordinate': 49.0, 'demand': 10, 'time_window_opening': 161.0, 'time_window_closing': 171.0, 'service_time': 10.0}, {'id': 1, 'x_coordinate': 35.0, 'y_coordinate': 17.0, 'demand': 7, 'time_window_opening': 50.0, 'time_window_closing': 60.0, 'service_time': 10.0}, {'id': 2, 'x_coordinate': 55.0, 'y_coordinate': 45.0, 'demand': 13, 'time_window_opening': 116.0, 'time_window_closing': 126.0, 'service_time': 10.0}, {'id': 3, 'x_coordinate': 55.0, 'y_coordinate': 20.0, 'demand': 19, 'time_window_opening': 149.0, 'time_window_closing': 159.0, 'service_time': 10.0}, {'id': 4, 'x_coordinate': 15.0, 'y_coordinate': 30.0, 'demand': 26, 'time_window_opening': 34.0, 'time_window_closing': 44.0, 'service_time': 10.0}, {'id': 5, 'x_coordinate': 25.0, 'y_coordinate': 30.0, 'demand': 3, 'time_window_opening': 99.0, 'time_window_closing': 109.0, 'service_time': 10.0}, 

In [233]:
def initialize_distance_matrix(data):
    points = [data['depot_coordinates']] + [(c['x_coordinate'], c['y_coordinate']) for c in data['customers']]
    num_of_points=len(points)
    matrix = [[0.0 for _ in range(num_of_points)] for _ in range(num_of_points)]
    for i in range(num_of_points):
        for j in range(num_of_points):
            x1,y1=points[i]
            x2,y2=points[j]
            matrix[i][j] = math.sqrt((x2 - x1)**2 + (y2 - y1)**2)
    return matrix
def get_demand(data):
    return [0] + [c['demand'] for c in data['customers']]
def initialize_pheromone_matrix(distance_matrix):
    m=len(distance_matrix)
    n=len(distance_matrix[0])
    pheromone_matrix = [[0.0 for _ in range(m)] for _ in range(n)]
    for i in range(m):
        for j in range(n):
            pheromone_matrix[i][j]=0.0001
    return pheromone_matrix
def initialize_visibility_matrix(distance_matrix):
    m=len(distance_matrix)
    n=len(distance_matrix[0])
    visibility_matrix = [[0.0 for _ in range(m)] for _ in range(n)]
    for i in range(m):
        for j in range(n):
            if i==j or distance_matrix[i][j]==0:
                visibility_matrix[i][j]=0
            else:
                visibility_matrix[i][j]=1.0/distance_matrix[i][j]
    return visibility_matrix
def probabilistic_traversal(distance_matrix,demand,max_capacity,Alpha,Beta,pheromone_matrix,visibility_matrix):
    pointer=0 #Depot
    length=len(distance_matrix)
    visited_status=[False for x in range(length)]
    visited_status[0]=True # Because that is the depot from where we start
    visited_node=[0]
    current_load=0
    while False in visited_status:
        feasible_indices=[]
        feasible_probabilities=[]
        current_node=distance_matrix[pointer]
        for j in range(length):
            if demand[j]+current_load<=max_capacity:
                if visited_status[j]==False:
                    thau = pheromone_matrix[pointer][j]
                    eta = visibility_matrix[pointer][j]
                    prob_numerator = (thau ** Alpha) * (eta ** Beta)
                    feasible_indices.append(j)
                    feasible_probabilities.append(prob_numerator)
        if feasible_indices:
            total=sum(feasible_probabilities)
            if total==0:
                selected_index=random.choice(feasible_indices)
            else:
                norm_probabilities = [p / total for p in feasible_probabilities]
                selected_index = random.choices(feasible_indices, weights=norm_probabilities, k=1)[0]
            visited_status[selected_index]=True
            current_load+=demand[selected_index]
            pointer=selected_index
            visited_node.append(pointer)
        else:
            if pointer!=0:
                pointer=0
                visited_node.append(pointer)
                current_load=0
            else:
                break
    if visited_node[-1]!=0:
        visited_node.append(0)
    return visited_node

def calculate_distance(distance_matrix, visited_node):
    total=0
    for i in range(len(visited_node)-1):
        from_node=visited_node[i]
        to_node=visited_node[i+1]
        total+=distance_matrix[from_node][to_node]
    return total
                    

                

In [234]:
def solve_basic_ACO():
    filename = "cvrp.txt"
    data = parse_fleet(filename)
    if not data:
        print("Failed to parse file.")
        return

    distance_matrix = initialize_distance_matrix(data)
    demands = get_demand(data)
    capacity = data['vehicle_capacity']
    
    length = len(distance_matrix)
    alpha=2.0
    beta = 3.5        # Priority for shorter distances
    rho = 0.1         # Evaporation rate
    num_ants = 10     # Ants per iteration
    iterations = 50   # Total generations

    pheromone_matrix = initialize_pheromone_matrix(distance_matrix)
    visibility_matrix = initialize_visibility_matrix(distance_matrix)
    
    # Champion Tracking
    best_global_dist = float('inf')
    best_global_tour = []

    print(f"Starting ACO for {length-1} customers...")

    # 4. Main Optimization Loop
    for it in range(iterations):
        all_tours = []
        all_distances = []

        # Step A: Let the Ants explore
        for ant_id in range(num_ants):
            # We call your traversal function
            tour = probabilistic_traversal(distance_matrix, demands, capacity,alpha, beta, 
                                    pheromone_matrix, visibility_matrix)
            dist = calculate_distance(distance_matrix, tour)
            
            all_tours.append(tour)
            all_distances.append(dist)
            #print(f"  Ant {ant_id}: Dist {dist:.2f} | Route: {tour}")

        # Step B: Identify the best ant of this generation
        min_dist = min(all_distances)
        best_tour = all_tours[all_distances.index(min_dist)]

        # Check if we found a new all-time best
        if min_dist < best_global_dist:
            best_global_dist = min_dist
            best_global_tour = best_tour
            print(f"Iteration {it}: Found shorter route! Distance: {best_global_dist:.2f}")

        # Step C: Pheromone Evaporation (The memory fades)
        for i in range(length):
            for j in range(length):
                pheromone_matrix[i][j] *= (1 - rho)

        # Step D: Pheromone Deposition (The best path gets reinforced)
        # We reward the best ant of THIS generation
        for k in range(len(best_tour) - 1):
            u, v = best_tour[k], best_tour[k+1]
            pheromone_matrix[u][v] += (1.0 / min_dist)

    # 5. Output Final Solution
    print("-" * 30)
    print("OPTIMIZATION COMPLETE")
    print(f"Best Total Distance: {best_global_dist:.2f}")
    print(f"Full Tour Sequence: {best_global_tour}")
solve_basic_ACO()

Starting ACO for 50 customers...
Iteration 0: Found shorter route! Distance: 818.60
Iteration 1: Found shorter route! Distance: 726.52
Iteration 2: Found shorter route! Distance: 726.52
Iteration 13: Found shorter route! Distance: 721.51
------------------------------
OPTIMIZATION COMPLETE
Best Total Distance: 721.51
Full Tour Sequence: [0, 46, 12, 5, 38, 11, 32, 1, 22, 28, 8, 26, 0, 47, 18, 6, 16, 50, 9, 49, 10, 15, 37, 0, 2, 20, 3, 31, 7, 24, 43, 23, 48, 0, 4, 40, 19, 17, 33, 45, 39, 30, 34, 21, 29, 35, 36, 0, 27, 14, 25, 13, 41, 44, 42, 0]


In [235]:
def solve_max_min_ACO():
    filename = "cvrp.txt"
    data = parse_fleet(filename)
    if not data:
        print("Failed to parse file.")
        return

    distance_matrix = initialize_distance_matrix(data)
    demands = get_demand(data)
    capacity = data['vehicle_capacity']
    
    length = len(distance_matrix)
    alpha=2.0
    beta = 3.5        # Priority for shorter distances
    rho = 0.1         # Evaporation rate
    num_ants = 10     # Ants per iteration
    iterations = 50   # Total generations

    pheromone_matrix = initialize_pheromone_matrix(distance_matrix)
    visibility_matrix = initialize_visibility_matrix(distance_matrix)
    
    # Champion Tracking
    best_global_dist = float('inf')
    best_global_tour = []

    print(f"Starting ACO for {length-1} customers...")

    # 4. Main Optimization Loop
    for it in range(iterations):
        all_tours = []
        all_distances = []
        # Step A: Let the Ants explore
        for ant_id in range(num_ants):
            # We call your traversal function
            tour = probabilistic_traversal(distance_matrix, demands, capacity,alpha, beta, 
                                    pheromone_matrix, visibility_matrix)
            dist = calculate_distance(distance_matrix, tour)
            
            all_tours.append(tour)
            all_distances.append(dist)
            #print(f"  Ant {ant_id}: Dist {dist:.2f} | Route: {tour}")

        # Step B: Identify the best ant of this generation
        min_dist = min(all_distances)
        best_tour = all_tours[all_distances.index(min_dist)]

        # Check if we found a new all-time best
        if min_dist < best_global_dist:
            best_global_dist = min_dist
            best_global_tour = best_tour
            print(f"Iteration {it}: Found shorter route! Distance: {best_global_dist:.2f}")

        # Step C: Pheromone Evaporation (The memory fades)
        for i in range(length):
            for j in range(length):
                pheromone_matrix[i][j] *= (1 - rho)

        # Step D: Pheromone Deposition (The best path gets reinforced)
        # We reward the best ant of THIS generation
        reward=1.0/best_global_dist
        for k in range(len(best_global_tour)-1):
            u,v=best_global_tour[k],best_global_tour[k+1]
            pheromone_matrix[u][v]+=reward
            pheromone_matrix[v][u]+=reward
            
        thau_max = 1.0 / (rho * best_global_dist)
        thau_min = thau_max / 20.0
        for i in range(length):
            for j in range(length):
                if pheromone_matrix[i][j] > thau_max:
                    pheromone_matrix[i][j] = thau_max
                elif pheromone_matrix[i][j] < thau_min:
                    pheromone_matrix[i][j] = thau_min

    # 5. Output Final Solution
    print("-" * 30)
    print("OPTIMIZATION COMPLETE")
    print(f"Best Total Distance: {best_global_dist:.2f}")
    print(f"Full Tour Sequence: {best_global_tour}")
solve_max_min_ACO()

Starting ACO for 50 customers...
Iteration 0: Found shorter route! Distance: 758.69
Iteration 1: Found shorter route! Distance: 757.15
Iteration 2: Found shorter route! Distance: 745.21
Iteration 3: Found shorter route! Distance: 708.31
Iteration 5: Found shorter route! Distance: 670.20
Iteration 10: Found shorter route! Distance: 642.64
Iteration 14: Found shorter route! Distance: 624.30
Iteration 25: Found shorter route! Distance: 624.30
Iteration 36: Found shorter route! Distance: 623.15
Iteration 42: Found shorter route! Distance: 623.15
Iteration 49: Found shorter route! Distance: 623.15
------------------------------
OPTIMIZATION COMPLETE
Best Total Distance: 623.15
Full Tour Sequence: [0, 46, 12, 5, 38, 16, 50, 21, 29, 2, 11, 0, 27, 48, 6, 18, 47, 4, 17, 37, 44, 15, 0, 9, 10, 30, 34, 20, 35, 36, 3, 22, 8, 0, 32, 1, 28, 31, 26, 7, 23, 43, 24, 25, 14, 0, 13, 40, 41, 19, 42, 45, 33, 39, 49, 0]
